<a href="https://colab.research.google.com/github/Cepera-dev/IA_Actividades/blob/main/Actividad_1_Angela_Estrategia_de_inversion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actividad 1 — Ángela — Estrategia de inversión
## NVIDIA (NVDA): de los apuntes a reglas contrastables
Versión de estudio. Cuatro alternativas: comprar y mantener, cruce de medias, reversión a la media y estrategia mixta 80/20. No contiene resultados históricos inventados: se generan al ejecutar las celdas en Colab.

### 1. Orden del material
| Fuente | Qué aporta | Cómo lo aplicamos |
|---|---|---|
| Clase 1: Buffett, Simons y mercado eficiente | Diferencia entre una explicación y una ventaja demostrada; sesgo de elegir ganadores | NVIDIA es un caso de estudio seleccionado hoy, no evidencia general |
| Clase 2: del indicador al modelo | SMA, RSI, MACD; señal retrasada y comparación sencilla | Reglas matemáticas antes de modelos complejos |
| Clase 3: datos | Splits, dividendos, huecos y disponibilidad temporal | OHLC ajustados, revisión y ninguna imputación con datos futuros |
| Clase 4: backtesting | Tendencia, reversión, momentum y arbitraje; entrada, salida, tamaño y riesgo | Comparar tendencia y reversión sobre el mismo activo y fechas |
| Clase 5: optimización | Fuerza bruta, gradiente, bayesiana y genéticos; fuera de muestra | Rejilla pequeña; elegir con entrenamiento y congelar antes del test |
| Ejercicio de Álvaro | Descarga, indicadores, curvas, métricas, optimización y reflexión | Conservar el esquema de seis pasos y reescribir reglas y evaluación |

### Qué revisar del ejemplo
La introducción anuncia un cruce de medias, pero la implementación y el informe usan precio/SMA con RSI. No son la misma regla. Incluye desplazamiento de señal en los retornos, pero las operaciones deben registrarse al precio de ejecución, no al cierre que produjo la señal. Añadimos costes y prueba temporal. Corregimos las definiciones: win rate = operaciones cerradas ganadoras / total de cerradas; Calmar = CAGR / magnitud del drawdown máximo. No trasladamos las conclusiones de Amazon a NVIDIA.

### 2. Objetivo e hipótesis
Pregunta: ¿puede una gestión activa mejorar el equilibrio entre rentabilidad y caídas de mantener NVIDIA? La estrategia mixta pregunta además si una caja de oportunidades aporta algo frente a mantener el 100 % invertido.
La tercera estrategia se inspira en la **familia de reversión a la media de la clase 4** y en la normalización estadística. Sus umbrales concretos son una propuesta propia, no una regla literal del profesor ni una propiedad demostrada de NVIDIA.

| Alternativa | Entrada | Salida |
|---|---|---|
| Comprar y mantener | Primera apertura del periodo | Último cierre del periodo |
| Tendencia SMA 10/200 | SMA10 > SMA200 | SMA10 ≤ SMA200 |
| Oportunidades por reversión | z ≤ −2 y cierre > SMA200 | z ≥ 0, cierre ≤ SMA200, pérdida al cierre ≥ 10 %, o 20 sesiones en posición |
| Mixta 80/20 | 80 % mantener y 20 % aplicar la reversión | Cada parte sigue sus reglas |

La regla de tendencia empieza invertida si ya existe tendencia al inicio: no exige esperar un nuevo cruce. Las señales se calculan al cierre y se ejecutan a la apertura siguiente. La condición de pérdida del 10 % es una **salida decidida al cierre**, no una orden stop intradía ni garantía de pérdida máxima: un gap puede empeorarla.

### La caja de oportunidades
Se reparten 100.000 USD iniciales: 80.000 núcleo y 20.000 oportunidades. No hay préstamos, cortos, transferencias ni rebalanceo entre partes. La caja conserva sus ganancias/pérdidas y puede estar totalmente en efectivo. Por tanto, **80/20 es el reparto inicial**, no un peso constante. Invertir todo el 20 % no equivale a arriesgar solo el 2 % del total: las pérdidas pueden exceder el umbral por gaps.

### Fundamento matemático
Para cierres ajustados P y ventana n: μ = media móvil de P, s = desviación móvil muestral de P; z = (P − μ)/s. z = −2 significa dos desviaciones por debajo de la media reciente. No implica una probabilidad gaussiana del 2,3 %, estacionariedad ni rebote asegurado. El filtro SMA200 evita algunas entradas contra tendencia, pero no garantiza que la reversión funcione.


## Cómo estudiar este cuaderno
**Lo principal es interpretar, no programar.** Lee el texto anterior a cada bloque, ejecuta su celda y responde a la pregunta de interpretación. El motor de cálculo es un bloque auxiliar que puedes plegar con la flecha de Colab.

Primera vuelta: entiende datos → reglas → operaciones → resultados. Segunda vuelta: entiende optimización y prueba fuera de muestra. No necesitas comprender todas las líneas de Python para explicar el ejercicio.

| Paso | Pregunta que debes poder contestar | Evidencia |
|---|---|---|
| 1. Objetivo | ¿Qué quiero mejorar frente a mantener? | Hipótesis escrita |
| 2. Datos | ¿Qué sabía al decidir y qué ajustes uso? | Histórico y controles |
| 3. Reglas | ¿Cuándo compro, vendo y cuánto invierto? | Tabla de reglas |
| 4. Backtest | ¿Qué habría pasado siguiendo esas reglas? | Operaciones, capital y caídas |
| 5. Optimización | ¿Qué parámetros elijo usando solo el pasado? | Rejilla de entrenamiento |
| 6. Evaluación | ¿Qué pasa en fechas que no elegí para ajustar? | Comparación fuera de muestra |
| 7. Reflexión | ¿Qué puedo afirmar y qué no? | Conclusiones respaldadas por cifras |

**Ejemplo de contabilidad:** de 100.000 USD, 80.000 permanecen invertidos y 20.000 esperan oportunidades. Si el núcleo gana un 10 % y la caja un 5 %, el valor final es 88.000 + 21.000 = 109.000 USD: retorno total del 9 %. La rentabilidad es ponderada por el capital, no la suma 10 % + 5 %. Es un ejemplo didáctico, no un resultado de NVIDIA.


In [ ]:
%pip install -q yfinance


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from IPython.display import display
from pathlib import Path
import json

TICKER = 'NVDA'
INICIO_DESCARGA = '2004-01-01'  # historia previa para las ventanas
INICIO_ESTUDIO = '2006-01-01'
FIN = '2026-11-16'  # exclusivo; evita incluir sesión parcial del día 16
INICIO_TEST = '2020-01-01'
CAPITAL = 100_000.0
COSTE = 0.001  # 0,10 % por compra y por venta, supuesto conjunto comisión/slippage
PESO_NUCLEO = 0.80
RF = 0.0  # efectivo y referencia Sharpe, simplificación explícita
SALIDA_PERDIDA = 0.10
MAX_SESIONES = 20
plt.style.use('seaborn-v0_8-whitegrid')
print('yfinance:', yf.__version__, '| pandas:', pd.__version__)


### 3. Datos y exploración
Se ajustan precios por splits y dividendos usando `auto_adjust=True`. El backtest usa unidades de precio ajustado como aproximación a una serie de retorno total; no es una contabilidad de acciones y dividendos de un bróker. Efectivo al 0 %, sin impuestos, cambio EUR/USD ni costes fijos. Los resultados se expresan en USD. El proveedor puede revisar el histórico: guarda el CSV para reproducir esta ejecución.
Si falla Yahoo, vuelve a ejecutar la descarga; no se sustituyen datos por simulaciones.


In [ ]:
df = yf.download(TICKER, start=INICIO_DESCARGA, end=FIN,
                 auto_adjust=True, multi_level_index=False, progress=False)
if df is None or df.empty:
    raise RuntimeError('Yahoo no devolvió datos. Reintenta la descarga.')
df = df.sort_index()
if df.index.tz is not None:
    df.index = df.index.tz_localize(None)
assert not df.index.duplicated().any(), 'Hay fechas duplicadas: revisar la fuente.'
assert df[['Open','Close']].notna().all().all(), 'Hay precios ausentes: revisar la fuente.'
assert (df[['Open','Close']] > 0).all().all(), 'Hay precios no positivos.'
print('Observaciones:', len(df), '| Desde:', df.index.min(), '| Hasta:', df.index.max())
print('Saltos de calendario > 4 días (inspeccionar; no son necesariamente errores):')
display(df.index.to_series().diff().loc[lambda x: x > pd.Timedelta(days=4)])
display(df[['Open','Close','Volume']].head())
fig, axes = plt.subplots(2,1,figsize=(12,7),sharex=True)
df.Close.plot(ax=axes[0],logy=True,title='NVIDIA: cierre ajustado, escala logarítmica')
df.Close.pct_change(fill_method=None).plot(ax=axes[1],title='Rentabilidad diaria')
plt.tight_layout(); plt.show()


**Cómo leer estos primeros gráficos:** la escala logarítmica permite comparar cambios relativos a lo largo de muchos años. En el gráfico de retornos, un pico de −0,10 representa una pérdida diaria del 10 %, no 10 dólares. Una gran subida histórica no demuestra que nuestra regla se hubiese podido elegir de antemano.

**Comprueba tu comprensión:** ¿por qué un split no debería aparecer como una pérdida económica del 90 %? Porque aumenta el número de acciones y reduce proporcionalmente el precio; el histórico ajustado evita interpretar ese cambio mecánico como pérdida.


In [ ]:
# Ver los indicadores sobre un tramo reciente: el gráfico explica las reglas.
precio=df.Close
media20=precio.rolling(20).mean()
desv20=precio.rolling(20).std(ddof=1)
z20=(precio-media20)/desv20.replace(0,np.nan)
tramo=df.index[-504:]
fig,axes=plt.subplots(2,1,figsize=(13,8),sharex=True)
axes[0].plot(tramo,precio.loc[tramo],label='Precio')
axes[0].plot(tramo,precio.rolling(10).mean().loc[tramo],label='SMA10')
axes[0].plot(tramo,precio.rolling(200).mean().loc[tramo],label='SMA200')
axes[0].plot(tramo,media20.loc[tramo],label='Media20',alpha=.7)
axes[0].fill_between(tramo,(media20-2*desv20).loc[tramo],(media20+2*desv20).loc[tramo],alpha=.12,label='Media20 ± 2 desviaciones')
axes[0].legend(); axes[0].set_title('Tendencia y distancia a la media: dos preguntas distintas')
axes[1].plot(tramo,z20.loc[tramo],label='z de 20 sesiones')
axes[1].axhline(-2,color='red',linestyle='--',label='Candidato a oportunidad')
axes[1].axhline(0,color='gray'); axes[1].legend()
plt.tight_layout(); plt.show()


**Idea clave:** el cruce pregunta «¿la tendencia reciente supera a la larga?». La reversión pregunta «¿el precio está inusualmente bajo respecto a su entorno reciente?». No son verdades contradictorias: estudian horizontes diferentes. Una caída puede profundizarse y una media móvil puede reaccionar tarde.


### 4. Motor de backtesting
Cada sesión: se ejecuta la orden pendiente en la apertura; se valora la cartera al cierre; se calcula la señal para la siguiente apertura. Compra con todo el efectivo de su parte, descontando el coste. Al final se liquida al cierre por convención común a todas las estrategias, con coste de venta. Cada test empieza sin posiciones, pero utiliza historia anterior para calcular indicadores. Se evita cobrar el movimiento nocturno anterior a una compra.


**Puedes plegar la siguiente celda.** Define una calculadora común para todas las alternativas. Lo importante es que aplica las mismas fechas, costes y reglas de ejecución. Un backtest es una simulación con supuestos, no una predicción.

Secuencia: cierre del lunes → calcular señal → apertura del martes → ejecutar. Si vendes en la apertura del martes, sí soportas el movimiento nocturno desde el lunes mientras seguías invertida; si compras el martes, no lo cobras retrospectivamente.


In [ ]:
def backtest(data, inicio, fin, tipo, capital=CAPITAL, coste=COSTE,
             corta=10, larga=200, ventana=20, umbral=2.0):
    p = data.Close
    sma_c, sma_l = p.rolling(corta).mean(), p.rolling(larga).mean()
    mu = p.rolling(ventana).mean()
    sigma = p.rolling(ventana).std(ddof=1).replace(0,np.nan)
    z = (p-mu)/sigma
    fechas = data.index[(data.index >= pd.Timestamp(inicio)) & (data.index < pd.Timestamp(fin))]
    if len(fechas)<2:
        raise ValueError('Periodo insuficiente')
    efectivo, unidades, pendiente = float(capital), 0.0, False
    precio_entrada, fecha_entrada, capital_entrada = None, None, None
    edad, filas, trades = 0, [], []
    def objetivo(t, invertido):
        if tipo == 'mantener': return True
        if pd.isna(sma_l.loc[t]): return False
        if tipo == 'medias': return bool(sma_c.loc[t] > sma_l.loc[t])
        if tipo != 'reversion': raise ValueError(tipo)
        if pd.isna(z.loc[t]): return False
        if not invertido:
            return bool(z.loc[t] <= -umbral and p.loc[t] > sma_l.loc[t])
        salir = (z.loc[t] >= 0 or p.loc[t] <= sma_l.loc[t]
                 or p.loc[t] <= precio_entrada*(1-SALIDA_PERDIDA) or edad >= MAX_SESIONES)
        return not salir
    pos = data.index.get_loc(fechas[0])
    if tipo == 'mantener': pendiente=True
    elif pos>0: pendiente=objetivo(data.index[pos-1],False)
    for i,t in enumerate(fechas):
        apertura, cierre = float(data.loc[t,'Open']), float(data.loc[t,'Close'])
        compra, venta, expuesto = False, False, False
        if unidades>0 and not pendiente:
            efectivo = unidades*apertura*(1-coste)
            trades.append(dict(entrada=fecha_entrada,salida=t,precio_entrada=precio_entrada,
                               precio_salida=apertura,pnl=efectivo-capital_entrada,
                               retorno=efectivo/capital_entrada-1,motivo='señal'))
            unidades=0.0; venta=True; edad=0
        elif unidades==0 and pendiente:
            capital_entrada=efectivo
            unidades=efectivo/(apertura*(1+coste)); efectivo=0.0
            precio_entrada,fecha_entrada=apertura,t
            compra=True; edad=0
        expuesto=unidades>0
        if expuesto: edad+=1
        pendiente=objetivo(t,expuesto)
        if i==len(fechas)-1 and unidades>0:
            efectivo=unidades*cierre*(1-coste)
            trades.append(dict(entrada=fecha_entrada,salida=t,precio_entrada=precio_entrada,
                               precio_salida=cierre,pnl=efectivo-capital_entrada,
                               retorno=efectivo/capital_entrada-1,motivo='fin del periodo'))
            unidades=0.0; venta=True
        filas.append(dict(fecha=t,capital=efectivo+unidades*cierre,
                          compra=compra,venta=venta,expuesto=expuesto))
    curva=pd.DataFrame(filas).set_index('fecha')
    curva['retorno']=curva.capital.pct_change(fill_method=None)
    curva.iloc[0,curva.columns.get_loc('retorno')]=curva.capital.iloc[0]/capital-1
    return curva,pd.DataFrame(trades)

def metricas(curva,trades,capital=CAPITAL):
    r=curva.retorno
    # Incluir el capital inicial en los máximos permite contar la pérdida inicial.
    maximos=curva.capital.cummax().clip(lower=capital)
    dd=curva.capital/maximos-1
    cagr=(curva.capital.iloc[-1]/capital)**(252/len(curva))-1
    vol=r.std(ddof=1)*np.sqrt(252)
    sharpe=(r.mean()-((1+RF)**(1/252)-1))/r.std(ddof=1)*np.sqrt(252) if r.std(ddof=1)>0 else np.nan
    pnl=trades.pnl if not trades.empty else pd.Series(dtype=float)
    ganancias,perdidas=pnl.clip(lower=0).sum(),-pnl.clip(upper=0).sum()
    return dict(Retorno=curva.capital.iloc[-1]/capital-1,CAGR=cagr,Volatilidad=vol,
                Sharpe=sharpe,Drawdown=dd.min(),Calmar=cagr/abs(dd.min()) if dd.min()<0 else np.nan,
                Operaciones_cerradas=len(trades),Win_rate=(pnl>0).mean() if len(pnl) else np.nan,
                Profit_factor=ganancias/perdidas if perdidas>0 else (np.inf if ganancias>0 else np.nan),
                Exposicion=curva.expuesto.mean())

def comparar(inicio,fin,parametros=(20,2.0),coste=COSTE):
    curvas, registros, resumen = {},{},{}
    for nombre,tipo in [('Comprar y mantener','mantener'),('Cruce SMA 10/200','medias'),('Reversión','reversion')]:
        c,t=backtest(df,inicio,fin,tipo,coste=coste,ventana=parametros[0],umbral=parametros[1])
        curvas[nombre],registros[nombre]=c,t
        resumen[nombre]=metricas(c,t)
    mixta=curvas['Comprar y mantener'].copy()
    mixta['capital']=PESO_NUCLEO*curvas['Comprar y mantener'].capital+(1-PESO_NUCLEO)*curvas['Reversión'].capital
    mixta['retorno']=mixta.capital.pct_change(fill_method=None)
    mixta.iloc[0,mixta.columns.get_loc('retorno')]=mixta.capital.iloc[0]/CAPITAL-1
    # Exposición: fracción efectiva invertida de la cartera, aproximada por parte.
    invertido=(PESO_NUCLEO*curvas['Comprar y mantener'].capital*curvas['Comprar y mantener'].expuesto+
               (1-PESO_NUCLEO)*curvas['Reversión'].capital*curvas['Reversión'].expuesto)
    mixta['expuesto']=invertido/mixta.capital
    t1=registros['Comprar y mantener'].copy(); t1['pnl']*=PESO_NUCLEO
    t2=registros['Reversión'].copy()
    if not t2.empty: t2['pnl']*=1-PESO_NUCLEO
    tm=pd.concat([t1,t2],ignore_index=True)
    curvas['Mixta 80/20']=mixta; registros['Mixta 80/20']=tm
    resumen['Mixta 80/20']=metricas(mixta,tm)
    # Las estadísticas por operación de la mixta mezclarían horizontes y tamaños distintos.
    for clave in ['Win_rate','Profit_factor']:
        resumen['Mixta 80/20'][clave]=np.nan
    return curvas,registros,pd.DataFrame(resumen).T


### 5. Optimización limitada al pasado
Entrenamiento 2006–2019; test 2020 hasta el final elegido. La separación es retrospectiva: no convierte esta investigación en una prueba prospectiva. Solo se optimiza la caja de oportunidades, con nueve combinaciones y Sharpe neto. Se exige un mínimo de diez operaciones cerradas en entrenamiento como filtro básico, no como garantía estadística. SMA10/200, filtro SMA200, salida del 10 %, máximo 20 sesiones y pesos 80/20 permanecen fijos. El 80/20 expresa una decisión de diseño, no un óptimo estimado.


In [ ]:
busqueda=[]
for ventana in [15,20,30]:
    for umbral in [1.5,2.0,2.5]:
        c,t=backtest(df,INICIO_ESTUDIO,INICIO_TEST,'reversion',ventana=ventana,umbral=umbral)
        busqueda.append(dict(ventana=ventana,umbral=umbral,**metricas(c,t)))
rejilla=pd.DataFrame(busqueda)
display(rejilla.sort_values('Sharpe',ascending=False))
candidatos=rejilla.loc[(rejilla.Operaciones_cerradas>=10)&rejilla.Sharpe.notna()]
if candidatos.empty:
    elegidos=(20,2.0)
    print('Sin candidatos suficientes: se mantiene la regla inicial 20/2, sin afirmar optimización.')
else:
    mejor=candidatos.sort_values(['Sharpe','ventana','umbral'],ascending=[False,True,True]).iloc[0]
    elegidos=(int(mejor.ventana),float(mejor.umbral))
print('Parámetros congelados antes de evaluar test:',elegidos)
mapa=rejilla.pivot(index='ventana',columns='umbral',values='Sharpe')
fig,ax=plt.subplots(figsize=(6,4))
im=ax.imshow(mapa.values,aspect='auto'); fig.colorbar(im,ax=ax,label='Sharpe entrenamiento')
ax.set_xticks(range(len(mapa.columns)),mapa.columns); ax.set_yticks(range(len(mapa.index)),mapa.index)
ax.set(xlabel='Umbral |z|',ylabel='Ventana',title='¿Máximo aislado o zona estable?')
plt.show()


**Cómo interpretar la optimización:** cada fila es una regla probada en entrenamiento. Un Sharpe máximo aislado puede ser suerte; varias combinaciones vecinas con cifras similares sugieren menor sensibilidad, sin demostrar una ventaja. Nueve pruebas ya implican selección: no podemos tratar la mejor como si se hubiese elegido antes de observar los datos.

**Pregunta:** ¿por qué no ajustamos con 2020–2026? Porque después dejaría de ser un periodo independiente de evaluación. La rejilla busca parámetros; no cambia la hipótesis para justificar el resultado.


### 6. Resultados fuera de muestra y evidencias
Las cuatro curvas usan el mismo periodo y capital. La mixta es la suma de sus dos partes, no un promedio de sus rentabilidades porcentuales acumuladas. Que reduzca caídas no implica que gane más. Win rate de mantener puede resultar 0 % o 100 % al liquidar: es una sola operación y no evidencia de capacidad predictiva.


In [ ]:
curvas,trades,tabla=comparar(INICIO_TEST,FIN,elegidos)
display(tabla.round(4))
fig,axes=plt.subplots(2,1,figsize=(13,9),sharex=True)
for nombre,c in curvas.items():
    axes[0].plot(c.index,c.capital,label=nombre)
    dd=c.capital/c.capital.cummax().clip(lower=CAPITAL)-1
    axes[1].plot(c.index,100*dd,label=nombre)
axes[0].set(yscale='log',ylabel='Capital USD',title='Comparación fuera de muestra, costes incluidos')
axes[1].set(ylabel='Drawdown %')
axes[0].legend(); plt.tight_layout(); plt.show()

c=curvas['Reversión']; precios=df.loc[c.index]
fig,ax=plt.subplots(figsize=(13,5))
ax.plot(precios.index,precios.Close,label='Cierre ajustado')
t=trades['Reversión']
if not t.empty:
    ax.scatter(t.entrada,t.precio_entrada,marker='^',color='green',label='Compra: apertura')
    ax.scatter(t.salida,t.precio_salida,marker='v',color='red',label='Venta: apertura o liquidación final')
ax.set_title('Oportunidades: precios de ejecución'); ax.legend(); plt.show()
display(t)

partes=pd.DataFrame({'Núcleo':PESO_NUCLEO*curvas['Comprar y mantener'].capital,
                    'Oportunidades':(1-PESO_NUCLEO)*curvas['Reversión'].capital})
display(partes.tail())
print('Peso final núcleo:',partes['Núcleo'].iloc[-1]/partes.sum(axis=1).iloc[-1])
print('Sesiones de la caja en efectivo:',1-curvas['Reversión'].expuesto.mean())


### Diccionario para interpretar los resultados
| Métrica | Interpretación | Precaución |
|---|---|---|
| Retorno | Ganancia acumulada respecto al capital inicial | Depende de la duración |
| CAGR | Tasa anual compuesta equivalente | No es una ganancia anual constante |
| Volatilidad | Dispersión de retornos diarios, anualizada | Incluye subidas y bajadas |
| Sharpe | Media de exceso de retorno / volatilidad | Aquí referencia 0 %; no es probabilidad de éxito |
| Drawdown | Mayor caída desde un máximo anterior | No indica que sea la peor caída futura |
| Calmar | CAGR / magnitud del drawdown | Depende mucho del periodo |
| Win rate | Fracción de operaciones cerradas con ganancia neta | Acertar mucho puede convivir con perder dinero |
| Profit factor | Ganancias monetarias / pérdidas monetarias | Sin pérdidas puede ser infinito y poco informativo |

En la tabla numérica, **0,25 equivale al 25 %** en retorno, CAGR, volatilidad, drawdown, win rate y exposición. Sharpe y Calmar son ratios. No elijas una estrategia mirando una sola columna.


### 7. Robustez y reflexión
Sensibilidad de costes con parámetros ya congelados. No usar el test para escoger nuevos umbrales o pesos. Si lo haces, pasa a ser validación y necesitarás otro test sin utilizar. Comparar también periodos bajistas y alcistas; repetir después sobre otros activos sin presentar la selección de NVIDIA como libre de sesgos.


In [ ]:
sensibilidad=[]
for coste in [0.0,0.001,0.003]:
    _,_,m=comparar(INICIO_TEST,FIN,elegidos,coste)
    for nombre,fila in m.iterrows():
        sensibilidad.append(dict(Estrategia=nombre,Coste_por_lado=coste,**fila.to_dict()))
display(pd.DataFrame(sensibilidad)[['Estrategia','Coste_por_lado','Retorno','Sharpe','Drawdown']])


### Guion para redactar la entrega
1. **Reglas:** explicar las cuatro alternativas y por qué tendencia y reversión son hipótesis diferentes.
2. **Datos:** fuente, fechas, ajustes, controles y limitaciones; captura del histórico.
3. **Implementación:** mostrar función de señales, estado de la posición y siguiente apertura.
4. **Backtest:** curvas, drawdown, métricas y registro de ejecuciones; capturas del test.
5. **Optimización:** tabla y mapa de entrenamiento; parámetros congelados y comparación posterior.
6. **Reflexión:** ¿mejora la mixta el Sharpe o el drawdown?, ¿qué retorno sacrifica?, ¿aporta la caja después de costes?, ¿cuántas operaciones sustentan la conclusión?, ¿es estable el entorno de parámetros?, ¿qué sesgos persisten?

No describir una estrategia como mejor antes de ejecutar y analizar sus resultados. Una caja a menudo en efectivo puede reducir riesgo simplemente por invertir menos. Como extensión útil, comparar la mixta con 80 % mantener y 20 % efectivo permanente para aislar la aportación activa. No se incluyen noticias ni sentimiento en estas reglas.

### Fuentes y alcance
Material del usuario: cinco clases 01–05 y notebook/informe de Álvaro. Reglas z-score y mezcla 80/20: diseño de Ángela para este ejercicio. Documentación técnica: https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html y https://pandas.pydata.org/docs/reference/window.html . La rejilla es un método de optimización; no es una estrategia de inversión. Markowitz sería una extensión de asignación entre activos, no una regla de entrada en una sola acción.


In [ ]:
# Referencia adicional: ¿aporta algo la caja frente a dejar ese 20 % inmóvil?
referencia=PESO_NUCLEO*curvas['Comprar y mantener'].capital+(1-PESO_NUCLEO)*CAPITAL
display(pd.DataFrame({'Mixta_activa':curvas['Mixta 80/20'].capital,
                      '80_mantener_20_efectivo':referencia}).tail())

salida=Path('resultados_angela'); salida.mkdir(exist_ok=True)
df.to_csv(salida/'NVDA_precios_ajustados.csv')
tabla.to_csv(salida/'metricas_test.csv')
rejilla.to_csv(salida/'optimizacion_entrenamiento.csv',index=False)
for nombre,c in curvas.items():
    seguro=nombre.replace(' ','_').replace('/','_')
    c.to_csv(salida/f'curva_{seguro}.csv')
    trades[nombre].to_csv(salida/f'operaciones_{seguro}.csv',index=False)
(salida/'configuracion.json').write_text(json.dumps(dict(ticker=TICKER,inicio=INICIO_ESTUDIO,
    inicio_test=INICIO_TEST,fin_exclusivo=FIN,coste=COSTE,peso_nucleo=PESO_NUCLEO,
    parametros_reversion=elegidos,yfinance=yf.__version__,pandas=pd.__version__),indent=2))
print('Resultados guardados en',salida,'; descárgalos desde el panel Archivos de Colab.')


### Lectura guiada de las conclusiones calculadas
La celda siguiente redacta observaciones a partir del test, sin inventar una explicación causal. «Mejor en este periodo» no equivale a «mejor para el futuro». La comparación 80/20 con efectivo sirve para comprobar si la actividad de la caja añade rentabilidad frente a no utilizarla.


In [ ]:
porcentaje=lambda x:f'{x:.2%}'
print('OBSERVACIONES DEL PERIODO DE PRUEBA')
for nombre in tabla.index:
    f=tabla.loc[nombre]
    print(f"{nombre}: retorno {porcentaje(f.Retorno)}, CAGR {porcentaje(f.CAGR)}, Sharpe {f.Sharpe:.2f}, caída máxima {porcentaje(f.Drawdown)}.")
mixta=tabla.loc['Mixta 80/20']; mantener=tabla.loc['Comprar y mantener']
print(f"La mixta cambia el retorno en {(mixta.Retorno-mantener.Retorno)*100:+.2f} puntos porcentuales frente a mantener.")
print(f"La mixta cambia la magnitud de la caída máxima en {(abs(mixta.Drawdown)-abs(mantener.Drawdown))*100:+.2f} puntos porcentuales; negativo significa una caída menor.")
print(f"Aportación final de operar la caja frente a 80 % mantener / 20 % efectivo: {curvas['Mixta 80/20'].capital.iloc[-1]-referencia.iloc[-1]:,.2f} USD.")
print(f"Operaciones cerradas de oportunidades en test: {len(trades['Reversión'])}.")
print('Estos resultados describen esta muestra; no prueban una ventaja futura ni separan por sí solos habilidad y azar.')


### Tus conclusiones: completar con criterio propio
- Mi objetivo principal era ______. La evidencia relevante es ______.
- La mixta obtiene ______ frente a mantener, a cambio de ______.
- Frente a 80 % mantener y 20 % efectivo, operar la caja aporta ______.
- Los parámetros elegidos con entrenamiento fueron ______; en test ocurrió ______.
- La limitación que más condiciona mi interpretación es ______.
- Antes de modificar las reglas, necesitaría comprobar ______.

**Qué no hace falta defender:** que la estrategia activa gane. Una conclusión académica sólida puede ser que no aporta suficiente mejora después de costes. Tampoco hace falta añadir IA a una actividad centrada en reglas técnicas y backtesting.
